# 08 - Morphotope-Level Characters

Following source `07_morphotope_chars.ipynb`:
Compute morphotope-level geometric features that go beyond building aggregation:
- `limLPS` — linear plan structure indicator
- `limAre/limPer/limLAL` — sums of top-10 connected building groups
- `limSDi` — spacing regularity via relative-neighborhood triangulation
- `limMFR/limMTC/limLMFR` — facade ratios and thin building ratios

In [1]:
import os
if os.path.basename(os.getcwd()).startswith("notebooks"):
    os.chdir(os.path.dirname(os.getcwd()))
print("Working dir:", os.getcwd())

import sys
sys.path.insert(0, "scripts")
from pipeline_config import load_city_config
load_city_config()

import numpy as np
import pandas as pd
import geopandas as gpd
import momepy as mm
from libpysal.graph import Graph, read_parquet
from shapely import unary_union
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

OUTPUT_ROOT = os.environ.get("OUTPUT_ROOT", "outputs")
PREPROCESSED_DIR = Path(OUTPUT_ROOT) / "01_preprocessed"
TESSELLATION_DIR = Path(OUTPUT_ROOT) / "02_tessellation"
GRAPH_DIR = Path(OUTPUT_ROOT) / "03_graphs"
MORPH_DIR = Path(OUTPUT_ROOT) / "06_morphotopes"
OUT = Path(OUTPUT_ROOT) / "08_morphotope_chars"
OUT.mkdir(parents=True, exist_ok=True)

Working dir: C:\urban-form\urban_taxonomy
[pipeline_config] City config: config\canada.env


In [2]:
tessellation = gpd.read_parquet(TESSELLATION_DIR / "tessellation.parquet")
buildings    = gpd.read_file(PREPROCESSED_DIR / "buildings.gpkg")
bq1          = read_parquet(GRAPH_DIR / "building_graph.parquet")
labels       = pd.read_parquet(MORPH_DIR / "tessellation_labels_postprocessed.parquet")["morphotope_label"]

# Map tessellation cells to morphotope labels
tessellation["morph"] = labels
print(f"{labels.nunique()} morphotopes")

12568 morphotopes


In [3]:
LIMIT_VALUE = 10  # top N connected building groups

def generate_ratio(group_idx, buildings, bq1):
    """Compute morphotope-level chars for a group of buildings (source logic)."""
    grp_buildings = buildings.loc[group_idx]
    if len(grp_buildings) < 3:
        return pd.Series({
            'limMFR': np.nan, 'limMTC': np.nan, 'limLMFR': np.nan,
            'limLPS': 0, 'limAre': grp_buildings.area.sum(),
            'limPer': grp_buildings.length.sum(),
            'limLAL': 0, 'limSDi': 0.35,
        })

    sub_bq1 = bq1.subgraph(group_idx)
    comp_labels = sub_bq1.component_labels

    # Union connected building groups
    connected = grp_buildings.geometry.groupby(comp_labels).apply(
        lambda x: unary_union(x.values)
    )
    connected = gpd.GeoSeries(connected, crs=buildings.crs).buffer(0.1).normalize().make_valid()

    fr = mm.facade_ratio(connected)
    elo = mm.elongation(connected)

    # "Thin" buildings: low facade ratio AND not too elongated
    thin = (fr < 8) & (elo < 0.90)
    areas = connected.area
    morph_fr_area_ratio = areas[thin].sum() / areas.sum() if areas.sum() > 0 else 0

    libNCos = grp_buildings["libNCo"].sum() if "libNCo" in grp_buildings.columns else 0
    limLPS = 1 if (morph_fr_area_ratio > 0.4 or libNCos > 0) else 0

    # Top-N sums
    top = min(LIMIT_VALUE, len(connected))
    sorted_areas = connected.area.sort_values(ascending=False)
    limAre = sorted_areas.iloc[:top].sum()
    sorted_pers = connected.length.sort_values(ascending=False)
    limPer = sorted_pers.iloc[:top].sum()
    lals = mm.longest_axis_length(connected)
    sorted_lals = lals.sort_values(ascending=False)
    limLAL = sorted_lals.iloc[:top].sum()

    # Spacing regularity
    try:
        rn = Graph.build_triangulation(
            grp_buildings.representative_point(), method="relative_neighborhood", kernel="identity"
        )
        dists = rn.aggregate("median")
        limSDi = dists.std() / grp_buildings.geometry.length.median() if grp_buildings.geometry.length.median() > 0 else 0
    except Exception:
        limSDi = 0.35

    return pd.Series({
        'limMFR': fr.median(), 'limMTC': thin.sum() / len(thin) if len(thin) > 0 else 0,
        'limLMFR': morph_fr_area_ratio, 'limLPS': limLPS,
        'limAre': limAre, 'limPer': limPer, 'limLAL': limLAL, 'limSDi': limSDi,
    })


# Group buildings by morphotope and compute chars
bldg_morph = buildings.merge(
    tessellation[["morph"]], left_index=True, right_index=True, how="inner"
)

morph_chars = bldg_morph.groupby("morph").apply(
    lambda g: generate_ratio(g.index, buildings, bq1), include_groups=False
)
print(f"Morphotope chars: {morph_chars.shape}")
print(morph_chars.describe().round(2))

Morphotope chars: (12568, 8)
        limMFR   limMTC  limLMFR    limLPS    limAre    limPer    limLAL  \
count  7195.00  7195.00  7195.00  12568.00  12568.00  12568.00  12568.00   
mean      2.62     0.87     0.88      0.56    735.66    198.57     62.48   
std       0.62     0.16     0.17      0.50   2138.96    242.50     90.64   
min       1.05     0.00     0.00      0.00      6.24     10.21      0.00   
25%       2.22     0.75     0.82      0.00    158.52     64.67      0.00   
50%       2.56     0.94     0.98      1.00    376.64    135.18     46.31   
75%       2.96     1.00     1.00      1.00    756.91    247.73     90.35   
max      12.90     1.00     1.00      1.00  77005.58   5484.29   1648.11   

         limSDi  
count  12568.00  
mean       0.42  
std        0.29  
min        0.00  
25%        0.32  
50%        0.35  
75%        0.41  
max        5.07  


In [4]:
morph_chars.to_parquet(OUT / "morph_chars.parquet")
print(f"Saved to {OUT / 'morph_chars.parquet'}")

Saved to outputs_canada\regions\4810\08_morphotope_chars\morph_chars.parquet
